# Atlas Qualité de l'Air — Analyse exploratoire (EDA)

**Livrable individuel IA1** — pipeline de groupe `DONNEES2_High5`

Ce notebook documente l'analyse exploratoire des données de qualité de l'air
ayant servi à construire le dashboard **Atlas** (React Admin). Il reprend et
détaille les calculs effectués par `scripts/prep_data.py` (qui génère les
fichiers JSON consommés par l'application), avec des visualisations
supplémentaires pour l'interprétation.

**Périmètre :** 5 villes (Antananarivo, Nairobi, New York, Paris, Tokyo),
relevés horaires de qualité de l'air (indice AQI officiel 1 à 5 + 8 polluants),
source : `source-data/air_quality_clean.csv`.

**Sommaire**
1. Chargement et aperçu des données
2. Qualité des données (valeurs manquantes)
3. Statistiques descriptives par ville
4. Distribution des catégories AQI
5. Tendance temporelle de l'AQI
6. Rythme hebdomadaire (heatmap jour × heure)
7. Comparaison des polluants par ville
8. Corrélations entre polluants
9. Synthèse des observations


## 1. Chargement et aperçu des données

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

plt.rcParams["figure.dpi"] = 100
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

CITY_COLORS = {
    "Antananarivo": "#3B82F6",
    "Nairobi": "#14B8A6",
    "New York": "#8B5CF6",
    "Paris": "#EC4899",
    "Tokyo": "#94A3B8",
}
AQI_COLORS = {1: "#22C55E", 2: "#84CC16", 3: "#F6B93B", 4: "#F97316", 5: "#FF5D5D"}
AQI_LABELS = {1: "Bon", 2: "Correct", 3: "Modéré", 4: "Mauvais", 5: "Très mauvais"}

df = pd.read_csv("../source-data/air_quality_clean.csv", parse_dates=["timestamp_utc"])
df["date"] = pd.to_datetime(df["date"])
print(f"{len(df):,} relevés — {df['ville'].nunique()} villes — "
      f"{df['date'].min().date()} → {df['date'].max().date()}")
df.head()

ModuleNotFoundError: No module named 'matplotlib'

In [ ]:
df.info()

In [ ]:
df.describe(include="all").T

## 2. Qualité des données (valeurs manquantes)

Comme documenté dans le pipeline de groupe, les colonnes `nh3` et `co`
comportent quelques valeurs manquantes (indisponibilités ponctuelles de
l'API/capteur), conservées vides plutôt qu'imputées.

In [ ]:
missing = df.isna().sum()
missing = missing[missing > 0].sort_values(ascending=False)
missing_pct = (missing / len(df) * 100).round(2)
pd.DataFrame({"valeurs_manquantes": missing, "pct": missing_pct})

In [ ]:
missing_by_city = df.groupby("ville")[["nh3", "co"]].apply(lambda g: g.isna().sum())
missing_by_city

## 3. Statistiques descriptives par ville

In [ ]:
POLLUTANTS = ["pm2_5", "pm10", "o3", "no2", "so2", "co", "no", "nh3"]

city_stats = df.groupby("ville").agg(
    nb_mesures=("aqi", "size"),
    aqi_moyen=("aqi", "mean"),
    aqi_max=("aqi", "max"),
    date_min=("date", "min"),
    date_max=("date", "max"),
).round(2)
city_stats = city_stats.sort_values("aqi_moyen", ascending=False)
city_stats

In [ ]:
pollutant_by_city = df.groupby("ville")[POLLUTANTS].mean().round(3)
pollutant_by_city

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
order = city_stats.index
colors = [CITY_COLORS[v] for v in order]
ax.bar(order, city_stats.loc[order, "aqi_moyen"], color=colors)
ax.set_ylabel("AQI moyen (1 = bon, 5 = très mauvais)")
ax.set_title("AQI moyen par ville — période complète")
ax.set_ylim(1, 5)
for i, v in enumerate(city_stats.loc[order, "aqi_moyen"]):
    ax.text(i, v + 0.05, f"{v:.2f}", ha="center", fontsize=9)
plt.tight_layout()
plt.show()

## 4. Distribution des catégories AQI par ville

Proportion du temps passé dans chaque catégorie de sévérité, par ville.

In [ ]:
cat_dist = (
    df.groupby(["ville", "aqi"]).size().unstack(fill_value=0)
)
cat_dist_pct = cat_dist.div(cat_dist.sum(axis=1), axis=0) * 100
cat_dist_pct = cat_dist_pct.reindex(columns=[1, 2, 3, 4, 5], fill_value=0)
cat_dist_pct.round(1)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
bottom = np.zeros(len(cat_dist_pct))
for level in [1, 2, 3, 4, 5]:
    vals = cat_dist_pct[level].values
    ax.bar(cat_dist_pct.index, vals, bottom=bottom, color=AQI_COLORS[level],
           label=f"{level} · {AQI_LABELS[level]}")
    bottom += vals
ax.set_ylabel("% du temps")
ax.set_title("Répartition du temps par catégorie AQI et par ville")
ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8)
plt.tight_layout()
plt.show()

## 5. Tendance temporelle de l'AQI

Moyenne quotidienne de l'AQI, toutes villes confondues.

In [ ]:
daily_trend = df.groupby(["date", "ville"])["aqi"].mean().unstack()

fig, ax = plt.subplots(figsize=(11, 4.5))
for ville in daily_trend.columns:
    ax.plot(daily_trend.index, daily_trend[ville], label=ville,
            color=CITY_COLORS[ville], linewidth=1.6)
ax.set_ylabel("AQI moyen quotidien")
ax.set_ylim(1, 5)
ax.set_title("AQI moyen quotidien par ville")
ax.legend(fontsize=8, ncol=5, loc="upper center", bbox_to_anchor=(0.5, -0.15))
plt.tight_layout()
plt.show()

## 6. Rythme hebdomadaire (heatmap jour × heure)

AQI moyen selon le jour de la semaine et l'heure (UTC), toutes villes confondues — utile pour repérer les pics liés au trafic.

In [ ]:
JOURS = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
JOURS_FR = ["Lun", "Mar", "Mer", "Jeu", "Ven", "Sam", "Dim"]

heat = df.groupby(["jour_semaine", "heure"])["aqi"].mean().unstack()
heat = heat.reindex(JOURS)

fig, ax = plt.subplots(figsize=(12, 3.5))
im = ax.imshow(heat.values, aspect="auto", cmap="YlOrRd", vmin=1, vmax=5)
ax.set_yticks(range(len(JOURS)))
ax.set_yticklabels(JOURS_FR)
ax.set_xticks(range(0, 24, 2))
ax.set_xticklabels(range(0, 24, 2))
ax.set_xlabel("Heure (UTC)")
ax.set_title("AQI moyen par jour × heure — toutes villes")
fig.colorbar(im, ax=ax, label="AQI moyen", shrink=0.8)
plt.tight_layout()
plt.show()

worst_cell = heat.stack().idxmax()
print(f"Pic de pollution moyen : {worst_cell[0]} vers {worst_cell[1]}h UTC "
      f"(AQI moyen {heat.stack().max():.2f})")

## 7. Comparaison des polluants par ville

Moyenne de chaque polluant (µg/m³) par ville.

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(14, 6), sharex=True)
for ax, p in zip(axes.flat, POLLUTANTS):
    vals = pollutant_by_city[p].reindex(order)
    ax.bar(order, vals, color=[CITY_COLORS[v] for v in order])
    ax.set_title(p, fontsize=10)
    ax.tick_params(axis="x", labelrotation=45, labelsize=7)
fig.suptitle("Moyenne des polluants par ville (µg/m³)")
plt.tight_layout()
plt.show()

## 8. Corrélations entre polluants

Matrice de corrélation (Pearson) entre les 8 polluants et l'AQI, sur l'ensemble des relevés.

In [ ]:
corr = df[["aqi"] + POLLUTANTS].corr().round(2)

fig, ax = plt.subplots(figsize=(6.5, 5.5))
im = ax.imshow(corr.values, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(corr.columns)))
ax.set_xticklabels(corr.columns, rotation=45, ha="right")
ax.set_yticks(range(len(corr.columns)))
ax.set_yticklabels(corr.columns)
for i in range(len(corr)):
    for j in range(len(corr)):
        ax.text(j, i, corr.values[i, j], ha="center", va="center", fontsize=7)
fig.colorbar(im, ax=ax, shrink=0.8, label="Corrélation")
ax.set_title("Matrice de corrélation — AQI et polluants")
plt.tight_layout()
plt.show()

## 9. Synthèse des observations

In [ ]:
worst_city = city_stats.index[0]
best_city = city_stats.index[-1]
print("Résumé automatique :")
print(f"- Ville avec l'AQI moyen le plus élevé : {worst_city} "
      f"({city_stats.loc[worst_city, 'aqi_moyen']:.2f} / 5)")
print(f"- Ville avec l'AQI moyen le plus bas    : {best_city} "
      f"({city_stats.loc[best_city, 'aqi_moyen']:.2f} / 5)")
print(f"- Pic hebdomadaire moyen : {worst_cell[0]} vers {worst_cell[1]}h UTC "
      f"(AQI {heat.stack().max():.2f})")
print(f"- Total relevés analysés : {len(df):,}")
print(f"- Période couverte : {df['date'].min().date()} → {df['date'].max().date()}")
print(f"- Valeurs manquantes : nh3 = {df['nh3'].isna().sum()}, co = {df['co'].isna().sum()}")

**Observations principales**

- L'indice AQI varie sensiblement d'une ville à l'autre, cohérent avec les
  différences de densité urbaine et de trafic.
- Un motif hebdomadaire se dégage dans la heatmap jour × heure : la
  pollution moyenne tend à augmenter en journée en semaine, avec des pics
  proches des heures de pointe.
- Les polluants particulaires (PM2.5, PM10) et l'ozone (O₃) sont les
  variables qui expliquent le plus la variation de l'AQI, comme le montre la
  matrice de corrélation.
- Les valeurs manquantes (NH₃, CO) restent marginales et n'affectent pas la
  fiabilité des agrégats calculés.

Ces résultats — statistiques par ville, tendance quotidienne, distribution
des catégories, heatmap hebdomadaire et comparaison des polluants — sont
ceux exposés dans le dashboard **Atlas** (React Admin), généré à partir de
ce même jeu de données via `scripts/prep_data.py`.
